In [4]:
# Exercise 1 Web Server Access Log Analysis

import csv
import json
import shutil
from pathlib import Path

# Setup student parameters
LAST_NAME = "Barquin"
STUDENT_ID = "TUPM-26-1221"  # Replace with your exact Student ID if different
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

# Data Generation Rules
RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_CLASS_NUM = (SEED_DIGIT % 4) + 2
TARGET_CLASS_STR = f"{TARGET_CLASS_NUM}xx"

# Paths
WORK_DIR = Path(".")
DATA_FILE = WORK_DIR / "data" / "weblog.csv"
OUTPUT_CSV = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_filtered_weblog.csv"
JSON_FILE = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"
BACKUP_JSON = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary_backup.json"

status_counts = {}
url_counts = {}
matched_records = []
processed_count = 0

with open(DATA_FILE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if processed_count >= RECORDS_TO_PROCESS:
            break
        processed_count += 1
        
        status = row.get("Staus", "").strip()
        url = row.get("URL", "").strip()
        ip = row.get("IP", "").strip()
        time = row.get("Time", "").strip()
        
        if status.startswith(str(TARGET_CLASS_NUM)):
            matched_records.append({"IP": ip, "Time": time, "URL": url, "Staus": status})
            status_counts[status] = status_counts.get(status, 0) + 1
            url_counts[url] = url_counts.get(url, 0) + 1

# Top 5 URLs
top_5_urls = sorted(url_counts.items(), key=lambda x: x[1], reverse=True)[:5]

# Export Filtered CSV
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["IP", "Time", "URL", "Staus"])
    writer.writeheader()
    writer.writerows(matched_records)

# Export JSON Summary
summary_data = {
    "records_processed": processed_count,
    "target_status_class": TARGET_CLASS_STR,
    "records_in_target_class": len(matched_records),
    "count_per_status_code": status_counts,
    "top_5_urls": top_5_urls
}

with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

# Create Backup
shutil.copy2(JSON_FILE, BACKUP_JSON)

print("--- EXERCISE 1 COMPLETE ---")
print("Records processed:", processed_count)
print("Target Class:", TARGET_CLASS_STR)
print("Matched Records:", len(matched_records))
print("Backup Verified:", BACKUP_JSON.exists())

--- EXERCISE 1 COMPLETE ---
Records processed: 1014
Target Class: 3xx
Matched Records: 164
Backup Verified: True


In [11]:
# Exercise 2: Mobile Device Usage Profiling

import csv
import json
import shutil
from pathlib import Path

LAST_NAME = "Barquin"
STUDENT_ID = "TUPM-26-1221"
SEED_DIGIT = int(STUDENT_ID[-1])

# Generation Rules
TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = str((SEED_DIGIT % 5) + 1)

# File Paths (points to the data folder)
WORK_DIR = Path(".")
DATA_FILE = WORK_DIR / "data" / "user_behavior_dataset.csv"
OUTPUT_CSV = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_users.csv"
JSON_FILE = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"
BACKUP_CSV = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_users_backup.csv"

total_records = 0
os_matched_count = 0
selected_users = []
total_battery = 0.0
total_data = 0.0

with open(DATA_FILE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_records += 1
        
        os_val = row.get("Operating System", "").strip()
        user_class = row.get("User Behavior Class", "").strip()
        
        if os_val.lower() == TARGET_OS.lower():
            os_matched_count += 1
            if user_class == TARGET_BEHAVIOR_CLASS:
                battery = float(row.get("Battery Drain (mAh/day)", 0))
                data_use = float(row.get("Data Usage (MB/day)", 0))
                
                total_battery += battery
                total_data += data_use
                
                selected_users.append({
                    "User ID": row.get("User ID"),
                    "Device Model": row.get("Device Model"),
                    "Operating System": os_val,
                    "Battery Drain (mAh/day)": battery,
                    "Data Usage (MB/day)": data_use
                })

count_selected = len(selected_users)
avg_battery = (total_battery / count_selected) if count_selected > 0 else 0.0
avg_data = (total_data / count_selected) if count_selected > 0 else 0.0

# Export CSV
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["User ID", "Device Model", "Operating System", "Battery Drain (mAh/day)", "Data Usage (MB/day)"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected_users)

# Export JSON
summary = {
    "operating_system": TARGET_OS,
    "user_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records": total_records,
    "os_matched_records": os_matched_count,
    "target_class_records": count_selected,
    "avg_battery_drain": round(avg_battery, 2),
    "avg_data_usage": round(avg_data, 2),
    "count_verification": count_selected == len(selected_users)
}

with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

# Backup & Size
shutil.copy2(OUTPUT_CSV, BACKUP_CSV)
backup_size = BACKUP_CSV.stat().st_size

print("--- EXERCISE 2 COMPLETE ---")
print(f"Target OS/Class: {TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}")
print("Selected Records:", count_selected)
print("Backup File Size:", backup_size, "bytes")



--- EXERCISE 2 COMPLETE ---
Target OS/Class: iOS, Class 2
Selected Records: 32
Backup File Size: 1082 bytes


In [12]:
# Exercise 3: Cybersecurity Incident Report

import csv
import json
import shutil
from pathlib import Path

LAST_NAME = "Barquin"
STUDENT_ID = "TUPM-26-1221"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

# Data Generation Rule
TARGET_YEAR = str(2015 + SEED_DIGIT)

# Paths
WORK_DIR = Path(".")
DATA_FILE = WORK_DIR / "data" / "Global_Cybersecurity_Threats_2015-2024.csv"
OUTPUT_CSV = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_incidents.csv"
JSON_FILE = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_top5.json"
BACKUP_JSON = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_top5_backup.json"
REPORT_TXT = WORK_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_report.txt"

total_records = 0
target_year_incidents = []
attack_counts = {}
total_loss = 0.0

with open(DATA_FILE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_records += 1
        
        # Parse Year
        year = row.get("Year", "").strip()
        if not year and "Timestamp" in row:
            year = row["Timestamp"].split("-")[0].split("/")[0]
            
        if year == TARGET_YEAR:
            attack_type = row.get("Attack Type", "Unknown").strip()
            try:
                loss = float(row.get("Financial Loss (in Million $)", 0))
            except ValueError:
                loss = 0.0
                
            row["parsed_loss"] = loss
            target_year_incidents.append(row)
            
            attack_counts[attack_type] = attack_counts.get(attack_type, 0) + 1
            total_loss += loss

incidents_count = len(target_year_incidents)
avg_loss = (total_loss / incidents_count) if incidents_count > 0 else 0.0

# Sort Top 5 by Loss descending
costliest_5 = sorted(target_year_incidents, key=lambda x: x["parsed_loss"], reverse=True)[:5]

# Remove temporary parsing key before export
for row in target_year_incidents:
    row.pop("parsed_loss", None)

# Export Filtered CSV
if target_year_incidents:
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=target_year_incidents[0].keys())
        writer.writeheader()
        writer.writerows(target_year_incidents)

# Export JSON
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(costliest_5, f, indent=4)

# Backup JSON
shutil.copy2(JSON_FILE, BACKUP_JSON)

# Text Report
with open(REPORT_TXT, "w", encoding="utf-8") as f:
    f.write(f"CYBERSECURITY INCIDENT REPORT ({TARGET_YEAR})\n")
    f.write("="*40 + "\n")
    f.write(f"Student: {LAST_NAME} ({STUDENT_ID})\n")
    f.write(f"Total Dataset Records: {total_records}\n")
    f.write(f"Incidents in {TARGET_YEAR}: {incidents_count}\n")
    f.write(f"Total Loss: ${total_loss:.2f} Million\n")
    f.write(f"Average Loss: ${avg_loss:.2f} Million\n\n")
    f.write("Incidents Per Attack Type:\n")
    for k, v in attack_counts.items():
        f.write(f"  - {k}: {v}\n")
    f.write("\nReport generated successfully.\n")

print("--- EXERCISE 3 COMPLETE ---")
print("Target Year:", TARGET_YEAR)
print("Incidents Count:", incidents_count)
print("Total Financial Loss: $", round(total_loss, 2), "Million")
print("Report Created:", REPORT_TXT.exists())

--- EXERCISE 3 COMPLETE ---
Target Year: 2016
Incidents Count: 285
Total Financial Loss: $ 13947.26 Million
Report Created: True
